# Kaggle Playground Series S6E8 — Predicting Smartphone Addiction

**Goal:** Build a practical, reproducible ROC-AUC solution and create `submission.csv`.

This notebook is designed to run **end-to-end without manual cell editing**. It uses CatBoost, handles missing categorical values safely, performs stratified cross-validation, adds a small set of behavior-based features, trains the final model, and writes the Kaggle submission file.

> Kaggle evaluates the probability of `addicted_label` using ROC-AUC, so the final submission contains probabilities, not 0/1 labels.


In [1]:
# 1. Imports
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from catboost import CatBoostClassifier

print("Libraries loaded.")


Libraries loaded.


In [2]:
# 2. Load data
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample_submission = pd.read_csv("sample_submission.csv")

print("Train:", train.shape)
print("Test :", test.shape)
print("Sample submission:", sample_submission.shape)
print("\nTarget distribution:")
print(train["addicted_label"].value_counts(normalize=True).rename("proportion"))


Train: (691369, 14)
Test : (296302, 13)
Sample submission: (296302, 2)

Target distribution:
addicted_label
1    0.709424
0    0.290576
Name: proportion, dtype: float64


In [3]:
# 3. Build features safely
# We remove 'id' from the model because it is only an identifier.
# We keep the original test IDs for the final submission.

TARGET = "addicted_label"
ID_COL = "id"

y = train[TARGET].copy()
train_ids = train[ID_COL].copy()
test_ids = test[ID_COL].copy()

X = train.drop(columns=[TARGET, ID_COL]).copy()
X_test = test.drop(columns=[ID_COL]).copy()

cat_cols = ["gender", "stress_level", "academic_work_impact"]

# IMPORTANT: do this BEFORE creating CV folds / X_train / X_valid.
# This prevents CatBoost's "bad object for id: nan" error.
for col in cat_cols:
    X[col] = X[col].fillna("Missing").astype(str)
    X_test[col] = X_test[col].fillna("Missing").astype(str)

# Behavior-based features.
def add_features(df):
    df = df.copy()

    # Ratios to daily screen time
    base = df["daily_screen_time_hours"].replace(0, np.nan)
    for col in ["social_media_hours", "gaming_hours", "work_study_hours", "weekend_screen_time"]:
        df[f"{col}_ratio"] = df[col] / base

    # Combined use and differences
    df["entertainment_hours"] = df["social_media_hours"] + df["gaming_hours"]
    df["non_work_screen_hours"] = df["daily_screen_time_hours"] - df["work_study_hours"]
    df["weekend_vs_daily"] = df["weekend_screen_time"] - df["daily_screen_time_hours"]

    return df

X = add_features(X)
X_test = add_features(X_test)

# CatBoost can handle missing numeric values, but replacing infinities from ratios
# with NaN keeps the data clean.
X = X.replace([np.inf, -np.inf], np.nan)
X_test = X_test.replace([np.inf, -np.inf], np.nan)

print("Final X shape:", X.shape)
print("Final test shape:", X_test.shape)
print("Categorical columns:", cat_cols)


Final X shape: (691369, 19)
Final test shape: (296302, 19)
Categorical columns: ['gender', 'stress_level', 'academic_work_impact']


In [4]:
# 4. Fast, reliable cross-validation
# Default = 3 folds for speed. Change to 5 if you have more time.
N_FOLDS = 5
RANDOM_STATE = 42

skf = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=RANDOM_STATE
)

oof_pred = np.zeros(len(X))
test_pred = np.zeros(len(X_test))
fold_scores = []
best_iterations = []

print(f"Starting {N_FOLDS}-fold Stratified CV...")


Starting 5-fold Stratified CV...


In [5]:
# 5. Train + validate
for fold, (train_idx, valid_idx) in enumerate(skf.split(X, y), start=1):
    print(f"\n{'='*55}")
    print(f"FOLD {fold}/{N_FOLDS}")
    print(f"{'='*55}")

    X_train = X.iloc[train_idx]
    X_valid = X.iloc[valid_idx]
    y_train = y.iloc[train_idx]
    y_valid = y.iloc[valid_idx]

    model = CatBoostClassifier(
        iterations=700,
        learning_rate=0.05,
        depth=7,
        loss_function="Logloss",
        eval_metric="AUC",
        cat_features=cat_cols,
        random_seed=RANDOM_STATE + fold,
        verbose=200,
        thread_count=-1,
        allow_writing_files=False
    )

    model.fit(
        X_train,
        y_train,
        eval_set=(X_valid, y_valid),
        early_stopping_rounds=80,
        use_best_model=True
    )

    # Probability of class 1 = addicted
    valid_pred = model.predict_proba(X_valid)[:, 1]
    test_fold_pred = model.predict_proba(X_test)[:, 1]

    # Store honest validation predictions
    oof_pred[valid_idx] = valid_pred

    # Average predictions from all CV models for the Kaggle test set
    test_pred += test_fold_pred / N_FOLDS

    fold_auc = roc_auc_score(y_valid, valid_pred)
    fold_scores.append(fold_auc)

    best_iter = model.get_best_iteration()
    best_iterations.append(best_iter)

    print(f"Fold {fold} ROC-AUC: {fold_auc:.6f}")
    print(f"Best iteration: {best_iter}")



FOLD 1/5
0:	test: 0.9104897	best: 0.9104897 (0)	total: 594ms	remaining: 6m 55s
200:	test: 0.9441451	best: 0.9441451 (200)	total: 1m 24s	remaining: 3m 30s
400:	test: 0.9512391	best: 0.9512391 (400)	total: 2m 47s	remaining: 2m 4s
600:	test: 0.9550408	best: 0.9550408 (600)	total: 4m 11s	remaining: 41.4s
699:	test: 0.9562059	best: 0.9562059 (699)	total: 4m 51s	remaining: 0us

bestTest = 0.9562058943
bestIteration = 699

Fold 1 ROC-AUC: 0.956206
Best iteration: 699

FOLD 2/5
0:	test: 0.9091097	best: 0.9091097 (0)	total: 424ms	remaining: 4m 56s
200:	test: 0.9451500	best: 0.9451500 (200)	total: 1m 37s	remaining: 4m 1s
400:	test: 0.9522905	best: 0.9522905 (400)	total: 3m 9s	remaining: 2m 21s
600:	test: 0.9558010	best: 0.9558010 (600)	total: 4m 44s	remaining: 46.8s
699:	test: 0.9569287	best: 0.9569287 (699)	total: 5m 29s	remaining: 0us

bestTest = 0.956928749
bestIteration = 699

Fold 2 ROC-AUC: 0.956929
Best iteration: 699

FOLD 3/5
0:	test: 0.9121648	best: 0.9121648 (0)	total: 503ms	remainin

In [6]:
# 6. Overall validation result
overall_auc = roc_auc_score(y, oof_pred)

print("\n" + "="*55)
print("CV RESULTS")
print("="*55)

for i, score in enumerate(fold_scores, 1):
    print(f"Fold {i}: {score:.6f}")

print(f"Mean fold ROC-AUC : {np.mean(fold_scores):.6f}")
print(f"Overall OOF ROC-AUC: {overall_auc:.6f}")
print(f"Mean best iteration: {np.mean(best_iterations):.0f}")



CV RESULTS
Fold 1: 0.956206
Fold 2: 0.956929
Fold 3: 0.957331
Fold 4: 0.958164
Fold 5: 0.957117
Mean fold ROC-AUC : 0.957149
Overall OOF ROC-AUC: 0.957148
Mean best iteration: 699


In [7]:
# 7. Save OOF predictions for project documentation
oof_results = pd.DataFrame({
    "id": train_ids,
    "actual": y,
    "oof_probability": oof_pred
})
oof_results.to_csv("oof_predictions.csv", index=False)

cv_results = pd.DataFrame({
    "fold": np.arange(1, N_FOLDS + 1),
    "roc_auc": fold_scores,
    "best_iteration": best_iterations
})
cv_results.to_csv("cv_results.csv", index=False)

print("Saved:")
print("- oof_predictions.csv")
print("- cv_results.csv")


Saved:
- oof_predictions.csv
- cv_results.csv


In [8]:
# 8. Final model
# Train once on ALL available training rows.
# We use the average best iteration from CV so the final model is not unnecessarily long.

final_iterations = int(max(100, round(np.mean(best_iterations))))

print(f"Training final CatBoost model with {final_iterations} iterations...")

final_model = CatBoostClassifier(
    iterations=final_iterations,
    learning_rate=0.05,
    depth=7,
    loss_function="Logloss",
    eval_metric="AUC",
    cat_features=cat_cols,
    random_seed=RANDOM_STATE,
    verbose=200,
    thread_count=-1,
    allow_writing_files=False
)

final_model.fit(X, y)

print("Final model trained.")


Training final CatBoost model with 699 iterations...
0:	total: 490ms	remaining: 5m 41s
200:	total: 1m 30s	remaining: 3m 44s
400:	total: 2m 57s	remaining: 2m 11s
600:	total: 4m 23s	remaining: 43s
698:	total: 5m 6s	remaining: 0us
Final model trained.


In [9]:
# 9. Generate final Kaggle predictions
final_test_pred = final_model.predict_proba(X_test)[:, 1]

submission = pd.DataFrame({
    "id": test_ids,
    "addicted_label": final_test_pred
})

submission.to_csv("submission.csv", index=False)

print("\nSubmission created successfully.")
print(submission.head())
print("\nPrediction range:")
print("min =", submission["addicted_label"].min())
print("max =", submission["addicted_label"].max())
print("mean =", submission["addicted_label"].mean())



Submission created successfully.
       id  addicted_label
0  691369        0.998729
1  691370        0.941616
2  691371        0.972445
3  691372        0.976002
4  691373        0.995768

Prediction range:
min = 0.0007567321924364604
max = 0.9999966916873837
mean = 0.7092497455073027


In [10]:
# 10. Final sanity checks
assert len(submission) == len(test), "Submission row count does not match test."
assert submission["id"].equals(test_ids), "Submission IDs do not match test IDs."
assert submission["addicted_label"].between(0, 1).all(), "Predictions must be probabilities between 0 and 1."

print("✅ Row count matches test.csv")
print("✅ IDs match test.csv")
print("✅ Predictions are valid probabilities")
print("\nFinal file ready: submission.csv")


✅ Row count matches test.csv
✅ IDs match test.csv
✅ Predictions are valid probabilities

Final file ready: submission.csv


## What to submit to Kaggle

Upload:

**`submission.csv`**

The submission has exactly the required columns:

`id, addicted_label`

The `addicted_label` column contains **probabilities** between 0 and 1, which is what the competition evaluates.

### Project files produced

- `submission.csv` — Kaggle submission
- `cv_results.csv` — fold-wise validation results
- `oof_predictions.csv` — out-of-fold predictions for analysis/documentation

### Kaggle submission workflow

1. Open the competition submission page.
2. Upload `submission.csv`.
3. Submit.
4. Record the public ROC-AUC score in your project README.

This competition uses ROC-AUC on the predicted probability for `addicted_label`. 
